In [48]:
import gymnasium as gym
import itertools
import math
import matplotlib
import matplotlib.colors as colors
import matplotlib.pyplot as plt
from matplotlib.tri import Triangulation
import random

import subprocess, sys

import numpy as np
import seaborn as sns

from tqdm import tqdm
import collections

# Make plots look nice
sns.set()
sns.set_context("notebook")
sns.set_style("whitegrid")

In [10]:
import ale_py
import gymnasium as gym

gym.register_envs(ale_py)


In [15]:
import gymnasium as gym

env = gym.make("ALE/Asterix-v5", obs_type="grayscale")

obs, info = env.reset(seed=42)

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

for _ in range(5):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"obs={obs}, reward={reward:.3f}, done={terminated}")

env.close()
print("✅ All good!")

Observation space: Box(0, 255, (210, 160), uint8)
Action space: Discrete(9)
obs=[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]], reward=0.000, done=False
obs=[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]], reward=0.000, done=False
obs=[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]], reward=0.000, done=False
obs=[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]], reward=0.000, done=False
obs=[[0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]], reward=0.000, done=False
✅ All good!


In [43]:
import tensorflow as tf
from tensorflow.keras import layers


inputs = tf.keras.Input(shape=(210, 160, 1))

x = layers.Conv2D(32, 3, activation='relu')(inputs)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(64, 3, activation='relu')(x)
x = layers.MaxPooling2D()(x)

x = layers.Conv2D(128, 3, activation='relu')(x)
x = layers.MaxPooling2D()(x)

x = layers.Flatten()(x)
x = layers.Dense(64, activation='relu')(x)

outputs = layers.Dense(9)(x) 

In [49]:
action_space = env.action_space
n_observations = 8
n_actions = 2
discount = 0.99

eps = 0.2

replay_buffer = []
optimizer = tf.keras.optimizers.Adam()


Q_net = tf.keras.Model(inputs, outputs)
T_net = tf.keras.Model(inputs, outputs)

for x in range(1000):

    obs, info = env.reset()
    obs = np.expand_dims(obs, axis=0)
    done = False

    if x % 10 == 0:
       T_net.set_weights(Q_net.get_weights())

    while not done:

        if(random.random()>eps):
            action = np.argmax(Q_net(obs, training = False).numpy())
        else:
            action = action_space.sample()


        next_obs, reward, terminated, truncated, info = env.step(action)
        next_obs = np.expand_dims(next_obs, axis=0)


        done = terminated or truncated

        replay_buffer.append([obs, action, reward, next_obs, terminated, truncated])
        replay_buffer = collections.deque(maxlen=10000)

        obs = next_obs

        Y = []
        batch = []
        actions = []

        if len(replay_buffer)>1000:

            replays = random.sample(replay_buffer, 32)

            for replay in replays:
                if(replay[4] or replay[5]):
                    y = replay[2]
                else:
                    y = replay[2] + discount * np.max(T_net(replay[3], training = False).numpy())

                Y.append(y)

                batch.append(replay[0])
                actions.append(replay[1])

            batch = np.vstack(batch)
            
            with tf.GradientTape() as tape:
                preds = tf.gather(Q_net(batch), actions, batch_dims=1)
                loss = tf.reduce_mean((preds - Y)**2)
            grads = tape.gradient(loss, Q_net.trainable_variables)
            optimizer.apply_gradients(zip(grads, Q_net.trainable_variables))

KeyboardInterrupt: 